<a href="https://colab.research.google.com/github/ElioRCDeepTech/proyecto-reproducible-elio/blob/main/NB4_Elio_Reproducibilidad_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB4 — Proyecto reproducible de aprendizaje automático
## Taller de reproducibilidad · Doctorado en Deep Tech · UNMSM

**Autor:** Elio Ramos Coronel  
**Correo:** elio.ramosc@unmsm.edu.pe

### Objetivo
Construir un proyecto que otra persona pueda clonar y ejecutar para obtener el mismo resultado, dentro del entorno de versiones fijadas. El cuaderno genera un repositorio Git, guarda el conjunto de datos, entrena un modelo con un `Pipeline` sin fuga de datos, registra cinco semillas en MLflow y prepara `README.md` y `requirements.txt`.

**Dataset del taller:** Wisconsin Breast Cancer, disponible en scikit-learn (569 observaciones, 30 variables predictoras y una etiqueta). El CSV se guarda dentro del repositorio para no depender de una descarga durante el entrenamiento.

**Criterios de entrega**
1. Cuatro o más commits con mensajes descriptivos.
2. Dos ejecuciones con la misma semilla y el mismo resultado.
3. Cinco semillas registradas en MLflow y reporte de media ± desviación estándar (DE).
4. README y dependencias fijadas para la prueba del extraño.

> Nota de integridad: reproducibilidad no demuestra por sí sola validez científica. La separación entrenamiento/prueba debe hacerse correctamente y cualquier transformación que aprenda parámetros debe estar dentro del `Pipeline`.


## Paso 0. Preparar el entorno

Configura tu identidad de Git y crea una carpeta de proyecto limpia. No escribas credenciales de GitHub en el cuaderno; el envío se solicita al final mediante una entrada oculta.

In [15]:
%pip -q install mlflow numpy pandas scikit-learn
import os, sys, json, hashlib, random, platform, subprocess, importlib.metadata
from pathlib import Path

AUTOR = "Elio Ramos Coronel"
CORREO = "elio.ramosc@unmsm.edu.pe"
PROYECTO = Path("/content/proyecto-reproducible-elio")
if PROYECTO.exists():
    import shutil
    shutil.rmtree(PROYECTO)
(PROYECTO / "src").mkdir(parents=True)
(PROYECTO / "data").mkdir()
os.chdir(PROYECTO)

subprocess.run(["git", "config", "--global", "user.name", AUTOR], check=True)
subprocess.run(["git", "config", "--global", "user.email", CORREO], check=True)
subprocess.run(["git", "config", "--global", "init.defaultBranch", "main"], check=True)
subprocess.run(["git", "init", "-q"], check=True)
print("Python:", platform.python_version())
print("Proyecto:", PROYECTO)
print("Autor configurado:", AUTOR, "|", CORREO)

Python: 3.13.16
Proyecto: /content/proyecto-reproducible-elio
Autor configurado: Elio Ramos Coronel | elio.ramosc@unmsm.edu.pe


## Paso 1. Guardar el dataset y su huella digital

El archivo de datos se versiona junto con el código. Su huella SHA-256 permite comprobar que el archivo no cambió. La carga inicial usa el dataset público incluido en scikit-learn; una vez guardado el CSV, las ejecuciones de entrenamiento leen ese archivo local.

In [3]:
from sklearn.datasets import load_breast_cancer
datos = load_breast_cancer(as_frame=True)
df = datos.frame.rename(columns={"target": "objetivo"})
ruta_datos = PROYECTO / "data" / "cancer_mama_wisconsin.csv"
df.to_csv(ruta_datos, index=False, float_format="%.17g")
huella_datos = hashlib.sha256(ruta_datos.read_bytes()).hexdigest()
print("Filas:", len(df), "| Predictores:", len(df.columns)-1)
print("SHA-256 del CSV:", huella_datos)

(PROYECTO / ".gitignore").write_text(
    "__pycache__/\n*.py[cod]\nmlruns/\nmlflow.db\n.ipynb_checkpoints/\n.env\n"
)
subprocess.run(["git", "add", ".gitignore", "data/cancer_mama_wisconsin.csv"], check=True)
subprocess.run(["git", "commit", "-m", "01 - Añadir dataset y reglas de exclusión"], check=True)


Filas: 569 | Predictores: 30
SHA-256 del CSV: 8164042552b007ddd10efc1aa66cd80a89936634beb60fe97a7b1f27f02ea5d5


CompletedProcess(args=['git', 'commit', '-m', '01 - Añadir dataset y reglas de exclusión'], returncode=0)

## Paso 2. Crear entrenamiento reproducible y sin fuga de datos

Se separan los datos en entrenamiento y prueba antes de ajustar cualquier transformación. `StandardScaler` está dentro de `Pipeline`; por tanto, en cada partición de validación cruzada aprende sus parámetros únicamente de la parte de entrenamiento. Se usa regresión logística como línea base y validación cruzada estratificada de cinco particiones.

La semilla controla la partición de validación cruzada y las fuentes de azar usadas por el script. `PYTHONHASHSEED` se establece al iniciar Python, no desde dentro del proceso.

In [7]:
codigo_entrenamiento = r'''import argparse, hashlib, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)

def entrenar(semilla=42, ruta_datos="data/cancer_mama_wisconsin.csv"):
    fijar_semilla(semilla)
    ruta = Path(ruta_datos)
    datos = pd.read_csv(ruta)
    X = datos.drop(columns="objetivo")
    y = datos["objetivo"].astype(int)
    pipeline = Pipeline([
        ("escalado", StandardScaler()),
        ("clasificador", LogisticRegression(max_iter=5000, random_state=semilla))
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
    puntuaciones = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=1)
    return {
        "semilla": int(semilla),
        "exactitud_media_cv": round(float(puntuaciones.mean()), 8),
        "exactitud_de_cv": round(float(puntuaciones.std(ddof=1)), 8),
        "huella_sha256_datos": hashlib.sha256(ruta.read_bytes()).hexdigest()
    }

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--semilla", type=int, default=42)
    parser.add_argument("--datos", default="data/cancer_mama_wisconsin.csv")
    args = parser.parse_args()
    print(json.dumps(entrenar(args.semilla, args.datos), ensure_ascii=False, sort_keys=True))
'''
(PROYECTO / "src" / "entrenar.py").write_text(codigo_entrenamiento, encoding="utf-8")
subprocess.run(["git", "add", "src/entrenar.py"], check=True)

# Evitar error si no hay cambios nuevos para hacer commit
try:
    subprocess.run(["git", "commit", "-m", "02 - Añadir Pipeline y entrenamiento sin fuga"], check=True)
except subprocess.CalledProcessError as e:
    # Si el error es solo que no hay nada para hacer commit, lo ignoramos
    status = subprocess.check_output(["git", "status"], text=True)
    if "nothing to commit" in status or "nada para hacer commit" in status:
        print("No hay cambios nuevos en entrenar.py para confirmar (commit ya realizado).")
    else:
        raise e

print("Script de entrenamiento guardado.")

No hay cambios nuevos en entrenar.py para confirmar (commit ya realizado).
Script de entrenamiento guardado.


### Comprobación de repetibilidad

Ejecuta dos veces la misma semilla. Los JSON deben ser idénticos. Si el resultado difiere, no continúes: revisa versiones, datos y entorno.

In [8]:
import os, subprocess, json
def ejecutar_semilla(semilla):
    entorno = os.environ.copy()
    entorno["PYTHONHASHSEED"] = "0"
    salida = subprocess.check_output(
        [sys.executable, "src/entrenar.py", "--semilla", str(semilla)],
        text=True, env=entorno
    ).strip()
    return salida

resultado_a = ejecutar_semilla(42)
resultado_b = ejecutar_semilla(42)
print("Primera ejecución:", resultado_a)
print("Segunda ejecución:", resultado_b)
assert resultado_a == resultado_b, "FALLO: la misma semilla produjo resultados diferentes"
print("PRUEBA SUPERADA: misma semilla = salida idéntica.")


Primera ejecución: {"exactitud_de_cv": 0.0185898, "exactitud_media_cv": 0.97366868, "huella_sha256_datos": "8164042552b007ddd10efc1aa66cd80a89936634beb60fe97a7b1f27f02ea5d5", "semilla": 42}
Segunda ejecución: {"exactitud_de_cv": 0.0185898, "exactitud_media_cv": 0.97366868, "huella_sha256_datos": "8164042552b007ddd10efc1aa66cd80a89936634beb60fe97a7b1f27f02ea5d5", "semilla": 42}
PRUEBA SUPERADA: misma semilla = salida idéntica.


## Paso 3. Registrar cinco semillas en MLflow

Cada ejecución registra la semilla, la huella de los datos, la versión de Git y las métricas. Después se informa la media ± DE de las cinco exactitudes medias obtenidas con semillas distintas. La DE se calcula entre las cinco ejecuciones (desviación estándar muestral, `ddof=1`).

In [9]:
import mlflow
import pandas as pd
from src.entrenar import entrenar

URI_MLFLOW = f"sqlite:///{(PROYECTO / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(URI_MLFLOW)
mlflow.set_experiment("NB4-Reproducibilidad-Elio")
commit_actual = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
semillas = [11, 23, 42, 77, 101]
resultados = []

for semilla in semillas:
    r = entrenar(semilla)
    resultados.append(r)
    with mlflow.start_run(run_name=f"semilla-{semilla}"):
        mlflow.log_params({
            "autor": AUTOR,
            "correo": CORREO,
            "modelo": "Regresión logística",
            "validacion": "Validación cruzada estratificada de 5 particiones",
            "semilla": semilla,
            "commit_git": commit_actual,
            "huella_sha256_datos": r["huella_sha256_datos"],
            "pipeline": "StandardScaler + LogisticRegression"
        })
        mlflow.log_metrics({
            "exactitud_media_cv": r["exactitud_media_cv"],
            "exactitud_de_cv": r["exactitud_de_cv"]
        })

tabla = pd.DataFrame(resultados).sort_values("semilla")
display(tabla)
media = tabla["exactitud_media_cv"].mean()
de = tabla["exactitud_media_cv"].std(ddof=1)
print(f"REPORTE: exactitud media = {media:.8f} ± {de:.8f} (DE muestral; 5 semillas)")
print("Experimento MLflow:", "NB4-Reproducibilidad-Elio")


2026/10/11 03:09:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/11 03:09:56 INFO mlflow.store.db.utils: Updating database tables
2026/10/11 03:09:59 INFO mlflow.tracking.fluent: Experiment with name 'NB4-Reproducibilidad-Elio' does not exist. Creating a new experiment.


,semilla,exactitud_media_cv,exactitud_de_cv,huella_sha256_datos
0,11,0.973638,0.018608,8164042552b007ddd10efc1aa66cd80a89936634beb60f...
1,23,0.980671,0.003906,8164042552b007ddd10efc1aa66cd80a89936634beb60f...
2,42,0.973669,0.018590,8164042552b007ddd10efc1aa66cd80a89936634beb60f...
3,77,0.975376,0.011509,8164042552b007ddd10efc1aa66cd80a89936634beb60f...
4,101,0.980671,0.015688,8164042552b007ddd10efc1aa66cd80a89936634beb60f...


REPORTE: exactitud media = 0.97680484 ± 0.00359849 (DE muestral; 5 semillas)
Experimento MLflow: NB4-Reproducibilidad-Elio


In [10]:
tabla.to_csv(PROYECTO / "resultados_5_semillas.csv", index=False)
subprocess.run(["git", "add", "resultados_5_semillas.csv"], check=True)
subprocess.run(["git", "commit", "-m", "03 - Registrar resumen de cinco semillas"], check=True)
print(subprocess.check_output(["git", "log", "--oneline"], text=True))


bcfed9e 03 - Registrar resumen de cinco semillas
be6fb4f 02 - Añadir Pipeline y entrenamiento sin fuga
0146a31 01 - Añadir dataset y reglas de exclusión



## Paso 4. Fijar dependencias y crear README

Las versiones se fijan explícitamente para reducir diferencias entre equipos. Para una prueba estricta, usa Python 3.11 y el mismo sistema operativo/arquitectura. El dataset CSV y su huella forman parte del repositorio. El README incorpora el resultado esperado calculado en esta ejecución y las instrucciones para comprobarlo.

In [11]:
versiones = {
    "numpy": importlib.metadata.version("numpy"),
    "pandas": importlib.metadata.version("pandas"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
    "mlflow": importlib.metadata.version("mlflow")
}
(PROYECTO / "requirements.txt").write_text(
    "".join(f"{paquete}=={version}\n" for paquete, version in versiones.items()),
    encoding="utf-8"
)
resultado_referencia = json.loads(ejecutar_semilla(42))
readme = f'''# Proyecto reproducible de ML — NB4 UNMSM

**Autor:** {AUTOR}
**Correo:** {CORREO}

## Objetivo
Demostrar repetibilidad y reproducibilidad mediante un pipeline de clasificación sin fuga de datos, cinco semillas registradas en MLflow y versiones de bibliotecas fijadas.

## Datos
Dataset Wisconsin Breast Cancer distribuido por scikit-learn y guardado como CSV en `data/cancer_mama_wisconsin.csv`.
- Filas: {len(df)}
- Variables predictoras: {len(df.columns)-1}
- Huella SHA-256: `{huella_datos}`

## Modelo y prevención de fuga
Regresión logística con `StandardScaler` dentro de `sklearn.pipeline.Pipeline`. Se emplea validación cruzada estratificada de 5 particiones. Cada transformación se ajusta dentro de cada partición de entrenamiento.

## Entorno
- Python recomendado: 3.11
- Dependencias exactas: ver `requirements.txt`
- Semillas del experimento: {semillas}

## Cómo reproducir
```bash
git clone URL_DEL_REPOSITORIO
cd proyecto-reproducible-elio
python -m venv .venv
# Linux/macOS:
source .venv/bin/activate
# Windows PowerShell: .venv\\Scripts\\Activate.ps1
python -m pip install --upgrade pip
pip install -r requirements.txt
PYTHONHASHSEED=0 python src/entrenar.py --semilla 42
```

En Windows PowerShell, ejecutar:
```powershell
$env:PYTHONHASHSEED="0"
python src/entrenar.py --semilla 42
```

## Resultado de referencia para semilla 42
La salida JSON esperada en este entorno es:
```json
{json.dumps(resultado_referencia, ensure_ascii=False, indent=2)}
```
La comparación debe hacerse con las mismas versiones de Python y bibliotecas, el CSV incluido y la misma arquitectura de ejecución. Si cambias versiones o plataforma, documenta la diferencia en vez de afirmar identidad exacta.

## Experimento de cinco semillas
Resultados: `resultados_5_semillas.csv`. Las ejecuciones se registran en MLflow local (`mlflow.db`, excluido de Git). Para consultar la interfaz:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Abre la dirección local que indique MLflow.

## Repetibilidad y límites
La misma semilla debe producir el mismo JSON dentro del entorno fijado. La media ± DE resume cinco particiones aleatorizadas; no constituye por sí sola una prueba de generalización externa ni elimina sesgos del dataset.
'''
(PROYECTO / "README.md").write_text(readme, encoding="utf-8")
subprocess.run(["git", "add", "requirements.txt", "README.md"], check=True)
subprocess.run(["git", "commit", "-m", "04 - Documentar entorno y prueba del extraño"], check=True)
print("requirements.txt:")
print((PROYECTO / "requirements.txt").read_text())
print("README.md creado.")
print(subprocess.check_output(["git", "log", "--oneline"], text=True))


requirements.txt:
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
mlflow==3.17.0

README.md creado.
0a2f222 04 - Documentar entorno y prueba del extraño
bcfed9e 03 - Registrar resumen de cinco semillas
be6fb4f 02 - Añadir Pipeline y entrenamiento sin fuga
0146a31 01 - Añadir dataset y reglas de exclusión



## Paso 5. Verificación final y envío opcional a GitHub

Esta celda verifica que existan al menos cuatro commits y que los archivos requeridos estén presentes. Para publicar, crea primero un repositorio vacío en GitHub. Después ejecuta el bloque de publicación y pega el enlace final en Moodle. El token se solicita de forma oculta y no se guarda en el notebook ni en el repositorio.

In [12]:
commits = subprocess.check_output(["git", "rev-list", "--count", "HEAD"], text=True).strip()
archivos_requeridos = ["README.md", "requirements.txt", "src/entrenar.py",
                       "data/cancer_mama_wisconsin.csv", "resultados_5_semillas.csv"]
faltantes = [f for f in archivos_requeridos if not (PROYECTO / f).exists()]
print("Número de commits:", commits)
print("Archivos faltantes:", faltantes)
assert int(commits) >= 4, "Se requieren al menos cuatro commits"
assert not faltantes, "Faltan archivos requeridos"
assert resultado_a == resultado_b, "La prueba de semilla no pasó"
print("VERIFICACIÓN LOCAL SUPERADA.")


Número de commits: 4
Archivos faltantes: []
VERIFICACIÓN LOCAL SUPERADA.


In [17]:
# Publicación opcional. Primero crea un repositorio vacío en GitHub.
PUBLICAR_EN_GITHUB = False
if PUBLICAR_EN_GITHUB:
    from getpass import getpass
    usuario_github = input("Nombre de usuario de GitHub: ").strip()
    token_github = getpass("Token de GitHub (entrada oculta): ")
    url_remota = f"https://{usuario_github}: {token_github}@github.com/{usuario_github}/proyecto-reproducible-elio.git"
    subprocess.run(["git", "remote", "add", "origin", url_remota], check=True)
    subprocess.run(["git", "push", "-u", "origin", "main"], check=True)
    print(f"Publicado: https://github.com/{usuario_github}/proyecto-reproducible-elio")
    del token_github
else:
    print("Publicación omitida. Cambia PUBLICAR_EN_GITHUB a True cuando hayas creado el repositorio vacío.")


Publicación omitida. Cambia PUBLICAR_EN_GITHUB a True cuando hayas creado el repositorio vacío.
